# Twitter Sentiment Analysis with NLP
## 1 Exploring dataset

#### 1.1 Load dataset

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import nltk

df = pd.read_csv('data/twitter.csv', encoding='utf-8')


#### 1.2 dataset info

In [ ]:
df.info()

#### 1.3 dataset top 5 rows

In [ ]:
df.head()

#### 1.4 Drop id column

In [ ]:
df.drop(columns=['id'], inplace=True)
df.head()

#### 1.5 Unique labels

In [ ]:
print(df['label'].nunique())
print(df['label'].value_counts())

# SNS plot to visualize
sns.countplot(x='label', data=df)
plt.show()

#### 1.6 Label Ratio

In [ ]:
df['label'].value_counts(normalize=True)

**Note**: The label class is imbalanced

#### 1.6 Check tweet length

In [ ]:
df['length'] = df['tweet'].apply(len)
df['length'].plot(bins=100, kind='hist')
plt.show()

## 2 Data cleanup
#### 2.1 Define preprocessor

In [ ]:
import re
import html
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()


def ensure_stopwords():
    try:
        nltk.data.find('corpora/stopwords')
    except LookupError:
        nltk.download('stopwords')

def ensure_wordnet():
    try:
        nltk.data.find('corpora/wordnet')
    except LookupError:
        nltk.download('wordnet')

ensure_stopwords()
ensure_wordnet()

stop_words = set(stopwords.words('english'))

def preprocessor(text):
    # Remove HTML tags
    text = html.unescape(text)
    text = re.sub(r'<[^>]*>', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Handle hashtags
    text = re.sub(r'#(\w+)', r'\1', text)

    # Extract Emoticons
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)

    # Normalize text
    text = text.lower()
    text = re.sub(r'[\W]+', ' ', text)

    # Remove non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    # Remove emoticons
    text = text + ' ' + ' '.join(emoticons).replace('-', '')

    # Remove stopwords
    tokens = [w for w in text.split() if w not in stop_words]

    tokens = [w for w in tokens if w.isalnum()]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return ' '.join(tokens)

#### 2.2 Test preprocessor

In [ ]:
text =  'In 1974, the teenager Martha Moxley (Maggie Grace) moves to the high-class area of Belle Haven, Greenwich, Connecticut. On the Mischief Night, eve of Halloween, she was murdered in the backyard of her house and her murder remained unsolved. Twenty-two years later, the writer Mark Fuhrman (Christopher Meloni), who is a former LA detective that has fallen in disgrace for perjury in O.J. Simpson trial and moved to Idaho, decides to investigate the case with his partner Stephen Weeks (Andrew Mitchell) with the purpose of writing a book. The locals squirm and do not welcome them, but with the support of the retired detective Steve Carroll (Robert Forster) that was in charge of the investigation in the 70\'s, they discover the criminal and a net of power and money to cover the murder.<br /><br />"Murder in Greenwich" is a good TV movie, with the true story of a murder of a fifteen years old girl that was committed by a wealthy teenager whose mother was a Kennedy. The powerful and rich family…<'

cleaned_text = preprocessor(text)
print(cleaned_text)

Test preprocessor on a Tweet

In [ ]:
row = df.loc[df['length'].idxmax()]

print(row['tweet'])

cleaned_text = preprocessor(row['tweet'])
print(cleaned_text)

#### 2.3 Apply preprocessor

In [ ]:
df['tweet'] = df['tweet'].apply(preprocessor)
print(df.loc[df['length'].idxmax()]['tweet'])

## 3 Sampling and dataset split
#### 3.1 Sampling

In [ ]:
def get_sampled_df(df, n_total, ratio, min_words=10, replace: bool = False):
    # keep only tweets with at least min_words tokens
    df = df[df["tweet"].astype(str).str.split().str.len() >= min_words]

    n_class1 = int(n_total * ratio)
    n_class0 = n_total - n_class1

    df_class0 = df[df['label'] == 0].sample(
        n=n_class0, random_state=42, replace=replace
    )
    df_class1 = df[df['label'] == 1].sample(
        n=n_class1,
        random_state=42,
        replace=replace
    )

    df_sampled = pd.concat([df_class0, df_class1]).sample(
        frac=1, random_state=42
    ).reset_index(drop=True)

    return df_sampled




# from imblearn.over_sampling import SMOTE
# from sklearn.feature_extraction.text import TfidfVectorizer
#
# vectorizer = TfidfVectorizer()
# X = vectorizer.fit_transform(df['tweet'])
# y = df['label']
#
# smote = SMOTE()
# X_res, y_res = smote.fit_resample(X, y)
#
# y_res.value_counts(normalize=True)

#### 3.1 Train test data split

In [ ]:
from sklearn.model_selection import train_test_split

# Sample dataset since label1 is minority
df_sampled = get_sampled_df(df, 3000, 0.3, min_words=15, replace=True)
df_sampled['label'].value_counts(normalize=True)

# X = df['tweet']
# y = df['label']

X = df_sampled['tweet']
y = df_sampled['label']

print(X.head())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)


## 4 Training
#### 4.1 Train baseline Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

def tokenizer(text):
    return text.split()

tfidf = TfidfVectorizer(
    strip_accents=None,
    lowercase=False,
    preprocessor=None,
    token_pattern=None
)

# Define pipeline
lr_tfidf = Pipeline([
    ('vect', tfidf),
    ('clf', LogisticRegression(solver='liblinear', random_state=0))
])

param_grid = [
    {
        'vect__ngram_range': [(1, 1)],  # Unigrams only
        'vect__stop_words': ['english', None],  # Stop words option
        'vect__tokenizer': [tokenizer],  # Your custom tokenizer
        'clf__penalty': ['l1', 'l2'],  # Penalty types
        'clf__C': [1.0, 10.0, 100.0]  # Regularization strength
    }
]

In [ ]:
from sklearn.model_selection import GridSearchCV

def tune_model(pipe, param_grid, X_train, y_train):
    # Create the GridSearchCV object
    best_model = GridSearchCV(pipe, param_grid, scoring='accuracy', cv=5, verbose=1, n_jobs=-1)

    # Fit the model to your training data (X_train, y_train)
    best_model.fit(X_train, y_train)

    return best_model



In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix

best_model = tune_model(lr_tfidf, param_grid, X_train, y_train)
# Evaluate the best model found by GridSearchCV
print("Best parameters: ", best_model.best_params_)
y_pred = best_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True)

In [ ]:
print(classification_report(y_test, y_pred))

## 5 Topic Modeling
#### 5.0 Define coherence evaluation method

In [ ]:
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel


def print_topic_coherence(
    model,
    documents,
    vectorizer=None,
    model_type="auto",
    coherence="c_v",
    top_n_words=10,
    max_topics=None,
    exclude_outlier=True
):
    # Convert documents to list of strings
    documents = [str(doc) for doc in documents]

    # Auto-detect model type
    if model_type == "auto":
        if hasattr(model, "get_topics"):
            model_type = "bertopic"
        elif hasattr(model, "components_"):
            model_type = "lda"
        else:
            raise ValueError("Could not detect model type. Use model_type='bertopic' or model_type='lda'.")

    if model_type == "bertopic":
        cleaned_docs = model._preprocess_text(documents)

        vectorizer = model.vectorizer_model
        analyzer = vectorizer.build_analyzer()

        tokens = [analyzer(doc) for doc in cleaned_docs]

        topics = model.get_topics().copy()
        if exclude_outlier:
          topics.pop(-1, None)

        topic_words = []

        for topic_id, words_scores in topics.items():
            if exclude_outlier and topic_id == -1:
                continue

            words = [
                word
                for word, score in words_scores[:top_n_words]
                if word != ""
            ]

            if words:
                topic_words.append(words)

            if max_topics is not None and len(topic_words) >= max_topics:
                break

    elif model_type == "lda":
        if vectorizer is None:
            raise ValueError("For sklearn LDA, you must pass the fitted CountVectorizer as vectorizer.")

        analyzer = vectorizer.build_analyzer()
        tokens = [analyzer(doc) for doc in documents]

        feature_names = vectorizer.get_feature_names_out()

        topic_words = []

        for topic in model.components_:
            top_word_indices = topic.argsort()[-top_n_words:][::-1]
            words = [feature_names[i] for i in top_word_indices]
            topic_words.append(words)

            if max_topics is not None and len(topic_words) >= max_topics:
                break

    else:
        raise ValueError("model_type must be 'auto', 'bertopic', or 'lda'.")

    dictionary = corpora.Dictionary(tokens)
    corpus = [dictionary.doc2bow(token) for token in tokens]

    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokens,
        corpus=corpus,
        dictionary=dictionary,
        coherence=coherence
    )

    coherence_score = coherence_model.get_coherence()

    print(f"Model type: {model_type}")
    print(f"Number of topics evaluated: {len(topic_words)}")
    print(f"Top words per topic: {top_n_words}")
    print(f"Coherence Score ({coherence}): {coherence_score}")

    return coherence_score

#### 5.1 LDA

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    stop_words='english',
    max_df=0.1, # we set the maximum document frequency of words to be considered to 10 percent
    max_features=5000, # we limited the number of words to be considered to the most frequently occurring 5,000 words
)

print(X_train.head())

X_vec = vectorizer.fit_transform(X)

lda = LatentDirichletAllocation(
    n_components=10,
    random_state=42,
    learning_method='batch'
)

X_topics = lda.fit_transform(X_vec)

lda.components_.shape

#### 5.1.1 Print/View topics

In [ ]:
def print_topics(model, vectorizer, n_top_words=5):
    feature_names = vectorizer.get_feature_names_out()

    for topic_idx, topic in enumerate(model.components_):
        top_word_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_word_indices]

        print(f"Topic {topic_idx + 1}:")
        print(" ".join(top_words))


#### 5.1.2 Print LDA

In [ ]:
print_topics(lda, vectorizer, n_top_words=5)
topic_cat1 = X_topics[:, 2].argsort()[::-1]


for iter_idx, movie_idx in enumerate(topic_cat1[:3]):
    print('\nTopic #%d:' % (iter_idx + 1))
    print(df['tweet'][movie_idx][:300], '...')

#### 5.1.3 Visualise topics

In [ ]:
!pip install pyLDAvis
import pyLDAvis
import pyLDAvis.lda_model

pyLDAvis.enable_notebook()

vis = pyLDAvis.lda_model.prepare(
    lda,
    X_vec,
    vectorizer
)

vis

#### 5.1.4 LDA Evaludation

In [ ]:
lda_coherence = print_topic_coherence(
    lda,
    X_train,
    vectorizer=vectorizer,
    top_n_words=10
)

#### 5.2 BERTopic

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_colab():
    !pip install bertopic
    !pip install sentence_transformers

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# embedding_model = SentenceTransformer(
#     "all-MiniLM-L6-v2",
#     device="cuda"
# )
#
# topic_model = BERTopic(embedding_model=embedding_model, language="english", calculate_probabilities=True, verbose=True)
topic_model = BERTopic(language="english", calculate_probabilities=True, verbose=True)
topics, probs = topic_model.fit_transform(X)

#### 5.2.1 Extracting topics

In [ ]:
freq = topic_model.get_topic_info(); freq.head(5)

**Select the most frequent topic**

In [ ]:
topic_model.get_topic(0)

#### 5.2.3 Visualisation

In [ ]:
topic_model.visualize_topics()

#### 5.2.4 Visualise Topic Probabilities

In [ ]:
topic_model.visualize_distribution(probs[200], min_probability=0.005)

#### 5.2.5 Visualise Topic Hierarchy

In [ ]:
topic_model.visualize_hierarchy(top_n_topics=50)

#### 5.2.6 Visualise Terms

In [ ]:
topic_model.visualize_barchart(top_n_topics=5)

#### 5.2.7 Visualise Topic Similarity

In [ ]:
topic_model.visualize_heatmap(n_clusters=5, width=1000, height=1000)

#### 5.2.8 Visualise Term Score Decline

In [ ]:
topic_model.visualize_term_rank()

#### 5.2.9 Model Evaluation

In [ ]:
topic_model_coherence = print_topic_coherence(
    topic_model,
    X,
    vectorizer=vectorizer,
    top_n_words=10
)